In [30]:
import pandas as pd 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from datetime import date
import time
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse


In [31]:
driver = webdriver.Chrome()
driver.get('https://www.flipkart.com/')
html_data = BeautifulSoup(driver.page_source, 'html.parser')

In [32]:
item_name = input('Enter the name of the product: ')
driver.get(f'https://www.flipkart.com/search?q={item_name}')
html_data = BeautifulSoup(driver.page_source, 'html.parser')
p = html_data.find_all('div',{'class':'_75nlfW'})

In [33]:
links = []
start_url = 'https://www.flipkart.com/'
for data in p:
    link = data.find('a', href=True) 
    main_url = link['href']
    links.append(start_url+main_url[1:])
links = links[0]

In [34]:
review_url = links.replace("/p/", "/product-reviews/")

In [35]:
#url

base_url = review_url

def modify_url(base_url):
    # Parse the base URL
    parsed_url = urlparse(base_url)
    query_params = parse_qs(parsed_url.query)
    
    # Modify query parameters to match the second URL format
    updated_params = {
        "pid": query_params.get("pid", [""])[0],
        "lid": query_params.get("lid", [""])[0],
        "aid": "overall",
        "certifiedBuyer": "false",
        "sortOrder": "MOST_HELPFUL"
    }
    
    # Create the updated query string
    updated_query = urlencode(updated_params)
    
    # Construct the new URL
    new_url = urlunparse((
        parsed_url.scheme,
        parsed_url.netloc,
        parsed_url.path,
        parsed_url.params,
        updated_query,
        parsed_url.fragment
    ))
    return new_url



# Convert the base URL
modified_url = modify_url(base_url)


In [36]:
reviews = ['MOST_HELPFUL','MOST_RECENT','POSITIVE_FIRST','NEGATIVE_FIRST']
index = modified_url[::-1].index('=')
modified_url = modified_url[:-index]



In [37]:
ratings = []
reviews_text = []
for review in reviews:
    url = modified_url + review
    i = 0
    while url != None:
        i += 1
        url = url + '&page=' + str(i)
        driver.get(url)
        html_data = BeautifulSoup(driver.page_source, 'html.parser')
        reviews = html_data.find_all('div',{'class':'EKFha-'})
        if not reviews:
            break
        for review in reviews:
            rating = review.find('div',{'class':'Ga3i8K'}).text
            ratings.append(rating)
            review_text = review.find('div',{'class':'ZmyHeo'}).text
            reviews_text.append(review_text)

In [38]:
data = pd.DataFrame({'ratings':ratings, 'review':reviews_text})
data.to_csv('reviews_and_comments.csv')